# Re-run Fase 6 (NFSP) amb criteri corregit — Google Colab

Executa el re-run del `best_nash` amb el fix (llindar 10M passos + 60 partides balancejades),
aprofitant la **GPU de Colab**. **No cal GitHub**: tot el codi (amb el fix) i els pesos
(`best_pesos_cos_truc.pth` + `best.zip`) van dins el fitxer **`tfg_rerun.zip`**.

**Abans de començar:**
1. Activa GPU: *Entorn d'execució → Canviar tipus d'entorn → T4 GPU*.
2. Puja `tfg_rerun.zip` a Google Drive, a la carpeta `MyDrive/TFG_rerun/`.
   (És a la teva carpeta de Descàrregues local.)

## 1. Comprovar GPU

In [ ]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CAP — activa GPU a Entorn d execucio!')

## 2. Muntar Drive i descomprimir el codi+pesos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
ZIP = '/content/drive/MyDrive/TFG_rerun/tfg_rerun.zip'
assert os.path.exists(ZIP), f'No trobo {ZIP} — puja tfg_rerun.zip a MyDrive/TFG_rerun/'

%cd /content
!rm -rf /content/TFG-truc && mkdir -p /content/TFG-truc
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/TFG-truc')
%cd /content/TFG-truc

# Verifica que el FIX hi és (ha de dir True) i que els pesos hi son
with open('RL/entrenament/entrenamentsComparatius/fase6/entrenament_fase6.py', encoding='utf-8') as f:
    print('Fix NASH_MIN_STEPS present:', 'NASH_MIN_STEPS' in f.read())
print('pesos_cos:', os.path.exists('RL/entrenament/entrenamentEstatTruc/registres/15_04_26_a_les_1217/models/best_pesos_cos_truc.pth'))
print('best.zip :', os.path.exists('TFG_Doc/notebooks/6_nfsp/resultats/ppo_nfsp/best.zip'))

## 3. Instal·lar dependències

In [ ]:
!pip install -q rlcard stable-baselines3 sb3-contrib gymnasium tqdm
print('Dependencies OK')

## 4. Llançar el re-run

Els defaults ja apliquen el fix (`NASH_MIN_STEPS=10M`, `sl_eval_sessions=60`). `--save_dir`
apunta a Drive, així que els resultats **sobreviuen a una desconnexió**.

**Recomanació:** fes primer una prova amb `--steps 2000000` (uns minuts) per confirmar que
arrenca bé; després puja-ho a `25000000` per al run definitiu (~2-4h a T4).

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/TFG_rerun/ppo_nfsp_rerun', exist_ok=True)
%cd /content/TFG-truc
!python RL/entrenament/entrenamentsComparatius/fase6/entrenament_fase6.py \
  --pesos_cos RL/entrenament/entrenamentEstatTruc/registres/15_04_26_a_les_1217/models/best_pesos_cos_truc.pth \
  --model_inicial TFG_Doc/notebooks/6_nfsp/resultats/ppo_nfsp/best.zip \
  --steps 2000000 \
  --num_envs 8 \
  --save_dir /content/drive/MyDrive/TFG_rerun/ppo_nfsp_rerun

## 5. Verificar resultats

El nou `best_nash.zip` (criteri corregit) queda a `MyDrive/TFG_rerun/ppo_nfsp_rerun/`.
Baixa la carpeta al repo local a `TFG_Doc/notebooks/6_nfsp/resultats/ppo_nfsp_rerun/` i el
notebook `comparacio_checkpoint3.ipynb` l'agafarà automàticament (columna `best_nash_rerun`).

In [ ]:
import os, pandas as pd
OUT = '/content/drive/MyDrive/TFG_rerun/ppo_nfsp_rerun'
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if os.path.isfile(p):
        print(f'{f:<24} {os.path.getsize(p):>12,} bytes')

df = pd.read_csv(os.path.join(OUT, 'training_log.csv'))
d = df[df.step >= 10_000_000]
if len(d):
    print('\nMillor exploit_vs_sl (>=10M):', round(d.exploit_vs_sl.min(), 2), 'pp al pas', int(d.loc[d.exploit_vs_sl.idxmin(), 'step']))
else:
    print('\n(Encara no s ha arribat al pas 10M; el best_nash es generara a partir d alla.)')